# Pilot 08 — Measurement-chain propagation (tutorial)

**Purpose.** Propagate an explicit sensor-calibration hypothesis *consistently* through the three
branches that share the environmental readings, and report it in a **common coordinate**.

Branches: (1) the **Ciddor anchor** deriving `n_1762` from the fringe-count ratio `R(t)`;
(2) the **predictors** fitting the measured coefficients; (3) the **Mathar predictions** used for
comparison. Perturbing one branch alone gives a misleading budget.

**Status:** pilot / scouting. Totals are provisional pending the statistical (nb 06) and within-campaign blocked
cross-validation (nb 01) repairs.

## 0. Calibration hypothesis — with a stated reference point

$$X_{\rm read}-X_0 = g_X\,(X_{\rm true}-X_0) + b_X
\qquad\Longrightarrow\qquad X_{\rm true}=X_0+\frac{X_{\rm read}-X_0-b_X}{g_X}$$

The reference point is not cosmetic: a gain-only **temperature** scan about 0 depends on Celsius vs
kelvin, and a **pressure** gain about 0 couples a slope change to a large baseline correction.
Bounds on `g` and `b` must jointly respect the assumed calibration envelope.

`R(t)` is an **instrument output** and is held **fixed** under any retrospective correction.

In [15]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '.')
from chain_propagation import (Models, Calibration, calibrate_K, propagate,
                               gain_scan, d_beta_identity, single_branch_estimate)

MODELS = '../models/'
DATA   = '../../data/processed/full_data.csv'
mo = Models(MODELS+'nist/refractive_index.py', MODELS+'mathar/Mathar2007.py')
df = pd.read_csv(DATA)

SUBSAMPLE = 1          # set to 1 for the production run
d = df.iloc[::SUBSAMPLE]
T, H, P, R, y = (d.temperature.values, d.humidity.values, d.pressure.values,
                 d.counts_ratio.values, d.n_1762.values)
print(f'rows: {len(d)} of {len(df)}')

rows: 145784 of 145784


## 1. `K` is a **reconstruction check**, not an independent calibration

`calibrate_K()` estimates `K` from the derived index whose construction we are auditing — it is
circular by design. Taking the mean can conceal row-dependent structure, so inspect

$$K_i=\frac{n_{1762,i}R_i}{n_{780,i}}$$

for spread and for environmental/temporal structure, and compare against the documented frequency
ratio. Verify the reconstructed index **row by row**, not only via one fitted coefficient.

⚠️ **Wavelength convention.** The anchor uses 780.24 nm and the documented ratio uses 1762.17 nm,
while Mathar is called at 1.762 µm. Keep *historical reproduction* distinct from a *consistently
specified* physical calculation.

In [16]:
n780 = mo.ciddor(780.24, T, P, H)
Ki = y * R / n780
print(f'K_i  mean {Ki.mean():.8f}   rel spread {Ki.std()/Ki.mean():.2e}   ptp/mean {(Ki.max()-Ki.min())/Ki.mean():.2e}')
for nm, v in [('T', T), ('H', H), ('P', P)]:
    print(f'  corr(K_i, {nm}) = {np.corrcoef(Ki, v)[0,1]:+.3f}')
lam = 1762.17/780.24
print(f'documented ratio {lam:.8f} vs recovered {Ki.mean():.8f}  -> {1e6*(Ki.mean()-lam)/lam:+.1f} ppm')
print('\nA vanishing spread means the archived index was generated by exactly this formula:')
print('the check confirms the toolset matches the pipeline, and cannot validate it independently.')
print('The ppm-level offset from the nominal wavelength ratio needs explaining before use.')

K_i  mean 2.25848570   rel spread 3.95e-13   ptp/mean 2.98e-12
  corr(K_i, T) = +0.913
  corr(K_i, H) = -0.511
  corr(K_i, P) = -0.122
documented ratio 2.25849739 vs recovered 2.25848570  -> -5.2 ppm

A vanishing spread means the archived index was generated by exactly this formula:
the check confirms the toolset matches the pipeline, and cannot validate it independently.
The ppm-level offset from the nominal wavelength ratio needs explaining before use.


In [17]:
K, _ = calibrate_K(mo, T, H, P, R, y)
base = propagate(mo, T, H, P, R, K)
print(f'd_alpha_H = {base.d_alpha_H:+.4e}   [archived full-data value: +4.3334e-09]')
print(f'offset-removed residual RMS   = {base.resid_offset_removed_rms:.4e} (RI units)')
print(f'fitted humidity-term RMS      = {base.fitted_H_term_rms:.4e} (RI units)')
print(f'  conditional H_perp form     = {base.fitted_H_term_rms_perp:.4e}')
print(f'  for scale only: {100*base.fitted_H_term_rms/base.resid_offset_removed_rms:.1f}%'
      f' of the total offset-removed RMS')
print('NOTE: |d_alpha_read|*std(H) is the RMS of the FITTED humidity term, NOT a variance')
print('      allocation - its fractional change equals the coefficient\'s by construction.')


d_alpha_H = +4.3334e-09   [archived full-data value: +4.3334e-09]
offset-removed residual RMS   = 1.9304e-07 (RI units)
fitted humidity-term RMS      = 1.6711e-08 (RI units)
  conditional H_perp form     = 1.3393e-08
  for scale only: 8.7% of the total offset-removed RMS
NOTE: |d_alpha_read|*std(H) is the RMS of the FITTED humidity term, NOT a variance
      allocation - its fractional change equals the coefficient's by construction.


### Algebraic check

$$\Delta\boldsymbol\beta_g=(X_g^{\mathsf T}X_g)^{-1}X_g^{\mathsf T}\left(\mathbf n_{{\rm data},g}-\mathbf n_{{\rm Mathar},g}\right)$$

evaluated by `lstsq` (better conditioned than the normal equations). This checks **algebraic
consistency only** — it does not validate the calibration convention, the model units, or the
physics.

In [18]:
ident = d_beta_identity(mo, T, H, P, R, K, Calibration())
print('max |identity - propagate| =', np.abs(ident - base.d_beta).max())

max |identity - propagate| = 8.152109782171403e-16


## 2. The common-coordinate slope is the primary comparison

Fitting against `H_g` rescales the slope's **units**. To compare across gain hypotheses, convert
back to a slope against the original readings:

$$\Delta\alpha_{H,\rm read}(g)=\frac{\Delta\alpha_{H,g}}{g}$$

Reporting `d_alpha_corrected` across different `g` compares quantities in different units.

The **intercept** transforms too, or the vector mixes coordinates:

$$\Delta\beta_{0,\rm read}=\Delta\beta_{0,g}+\sum_j\Delta\beta_{j,g}\left[X_{0j}(1-1/g_j)-b_j/g_j\right]$$

In [ ]:
base, rows_o = gain_scan(mo, T, H, P, R, K, [1.05,1.15,1.303,1.33], channel='H', uncentred=True)
_,    rows_c = gain_scan(mo, T, H, P, R, K, [1.05,1.15,1.303,1.33], channel='H', uncentred=False)
o, c = pd.DataFrame(rows_o), pd.DataFrame(rows_c)
cmp = pd.DataFrame({
    'g': o.g,
    'original b=(g-1)H0': o.d_alpha_read,
    'orig chg %':  100*(o.d_alpha_read/base.d_beta_read[2]-1),
    'centred b=0': c.d_alpha_read,
    'cent chg %':  100*(c.d_alpha_read/base.d_beta_read[2]-1),
    'consistent shift': o.d_alpha_read - base.d_beta_read[2],
    'SM single-branch (comparison only)': o.single_branch,
    'RMS chg % orig': 100*(o.resid_rms/base.resid_offset_removed_rms-1),
    'RMS chg % cent': 100*(c.resid_rms/base.resid_offset_removed_rms-1),
})
display(cmp)
print('The two scenarios coincide only when b = (g-1)*H0; equivalent_uncentred() supplies it.')
print('SM single-branch rescales ONLY the fitted data coefficient, alpha_H*(g-1) (signed;')
print('the SM quotes its magnitude):')
print(f'  at g=1.303: {o.single_branch[2]:+.3e}, while consistent three-branch propagation')
print(f'  moves the READ-coordinate difference by {o.d_alpha_read[2]-base.d_beta_read[2]:+.3e}')
print('  -> the single-branch treatment overstates the gain systematic by ~10x.')


## 3. Two things the aggregate cannot do

**(a) It cannot be quoted across scenarios.** The total offset-removed RMS changes by ~0.5 % under
the ORIGINAL hypothesis (b = (g−1)H₀) and ~0.002 % under the CENTRED b = 0 one. That is a changed
physical hypothesis, not a reporting choice.

**(b) The fitted humidity term is not a variance allocation.** `|Δα_read|·std(H)` has a fixed
multiplier, so its fractional change equals the coefficient's *by construction*. It confirms nothing
independently. The residualised form (H against 1, T, P) is a statistical decomposition only — not
identification of a physical noise source.

Note the residual is invariant under mere **reparameterisation**, but **not** under the physical
calibration hypothesis.

## 4. Design questions

Evaluate as questions, not conclusions. Report **sensitivity per unit input perturbation** alongside
results for our assumed tolerances, so a reader can substitute their own specifications.

**Q1** — how much does constraining `g_H` reduce uncertainty in the humidity comparison, with the
present statistical limitations retained?

**Q2** — does a better frequency reference materially improve coefficient discrimination or slow
compensation? (Enter as a perturbation on `K`.)

Label every source **quantified / bounded / unresolved**, with assumed magnitude, temporal
behaviour, evidence, and propagated effect.

In [20]:
for frac in (1e-9, 1e-8, 1e-7):
    r_k = propagate(mo, T, H, P, R, K*(1+frac))
    print(f'dK/K={frac:.0e}: d_alpha_H(read) {r_k.d_alpha_H_read:+.4e}  '
          f'resid RMS {r_k.resid_offset_removed_rms:.4e}  offset {r_k.resid_offset:+.3e}')

dK/K=1e-09: d_alpha_H(read) +4.3334e-09  resid RMS 1.9304e-07  offset +1.167e-06
dK/K=1e-08: d_alpha_H(read) +4.3334e-09  resid RMS 1.9304e-07  offset +1.176e-06
dK/K=1e-07: d_alpha_H(read) +4.3334e-09  resid RMS 1.9304e-07  offset +1.266e-06


## 5. What this notebook does **not** establish

- It does not measure the actual sensor gain; it computes what a *hypothesised* gain would do.
- It does not resolve the temporal-dependence uncertainty (nb 06). The paired difference is
  **not statistically robust across the resampling procedures examined** — carry that statement,
  not a single nominal significance figure.
- It does not address Mathar extrapolation (91.7 % of rows above 25 °C) or the limited in-domain
  coverage (~1.8 days equivalent, January only).
- A reduced gain sensitivity would **weaken the attribution to humidity gain**. It would *not*
  restore evidence for a physical anomaly.